# MKNN Alignment from HuggingFace embeddings

Runs three Aristotelian MKNN analyses using only the embeddings on
`nitrox639/platonic-embeddings`:

1. **Intramodal** alignment within each EEG model (consecutive size pairs). reve is skipped.
2. **Cross-modal** EEG vs the matching VideoMAE embeddings (k=5, k=10).
3. **Cross-modal** EEG vs the matching DINOv2 embeddings (k=5, k=10).

All embeddings are **layerwise**: EEG arrays are `(n_layers, W, S, D)` and the
VideoMAE / DINOv2 arrays are `(n_layers, W, D)`. Every analysis searches over
*all* layer pairs (EEG layer i vs vision/EEG layer j), reports the maximum mKNN,
and calibrates that maximum against an aggregation-aware permutation null — the
protocol of Huh et al. 2024 / the Aristotelian-view paper.

All helpers, constants, and the generic cross-modal driver `run_cross_modal(...)`
live in the setup cell. Subsequent cells just call `run_cross_modal(..., k_mknn)`
and plot the results.

In [ ]:
from pathlib import Path
import os, json
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors
from huggingface_hub import hf_hub_download

# ── Auth ──────────────────────────────────────────────────────────────────────
REPO = 'nitrox639/platonic-embeddings'
for _tp in (Path('../tokens/hf_token.txt'), Path('tokens/hf_token.txt'),
            Path('hf_token.txt'), Path.home() / 'hf_token.txt'):
    if _tp.exists():
        os.environ['HF_TOKEN'] = _tp.read_text().strip()
        break

def fetch(p):
    return hf_hub_download(REPO, p, repo_type='dataset')

def load_npz(p):
    return np.load(fetch(p), allow_pickle=True)

# ── Model registry ────────────────────────────────────────────────────────────
EEG = {
    'femba':       {'sizes': ['tiny', 'base', 'large'],  'family': 'femba_luna',
                    'fname': lambda s: f'eeg/femba/femba_{s}_tusl_layerwise.npz'},
    'luna':        {'sizes': ['base', 'large', 'huge'],  'family': 'femba_luna',
                    'fname': lambda s: f'eeg/luna/luna_{s}_layerwise.npz'},
    'neurolm':     {'sizes': ['b', 'l', 'xl'],           'family': 'neurolm',
                    'fname': lambda s: f'eeg/neurolm/neurolm_{s}_layerwise.npz'},
    'reve':        {'sizes': ['base', 'large'],          'family': 'reve',
                    'fname': lambda s: f'eeg/reve/reve_{s}_layerwise.npz'},
    'steegformer': {'sizes': ['small', 'base', 'large'], 'family': 'steegformer',
                    'fname': lambda s: f'eeg/steegformer/steegformer_{s}_layerwise.npz'},
}
VIDEOMAE_SIZES = ['base', 'large']
DINOV2_SIZES   = ['small', 'base', 'large', 'giant']
VJEPA2_SIZES   = ['large', 'huge', 'giant']

def vmpath(vs, fam):  return f'vision/videomae/videomae_{vs}__{fam}.npz'
def dnpath(vs, fam):  return f'vision/dinov2/dinov2_{vs}__{fam}.npz'
def vj2path(vs, fam): return f'vision/vjepa2/vjepa2_{vs}__{fam}.npz'

# ── Display metadata ──────────────────────────────────────────────────────────
PARAMS = {
    'femba':       {'tiny': 8,   'base': 48,  'large': 78},
    'luna':        {'base': 7,   'large': 43, 'huge': 311},
    'neurolm':     {'b': 254,    'l': 500,    'xl': 1696},
    'steegformer': {'small': 22, 'base': 86,  'large': 307},
    'reve':        {'base': 69,  'large': 408},
}
DISPLAY = {'femba': 'FEMBA', 'luna': 'LUNA', 'neurolm': 'NeuroLM',
           'steegformer': 'STEEGFormer', 'reve': 'REVE'}
def fmt_size(model, sz): return f'{sz} ({PARAMS[model][sz]}M)'

vm_colors  = {'base': 'steelblue', 'large': 'seagreen'}
dn_colors  = {'small': 'steelblue', 'base': 'seagreen',
              'large': 'darkorange', 'giant': 'mediumpurple'}
vj2_colors = {'large': 'steelblue', 'huge': 'seagreen', 'giant': 'darkorange'}

# ── Hyperparameters (mirror analyze_mknn.ipynb) ───────────────────────────────
K_MKNN = 5
K_PERM = 200
ALPHA  = 0.05

# ── kNN / mKNN primitives ─────────────────────────────────────────────────────
def l2(x):
    return x / (np.linalg.norm(x, axis=-1, keepdims=True) + 1e-12)

def knn_1d(z, k):
    nn = NearestNeighbors(n_neighbors=k+1, metric='cosine', algorithm='brute').fit(z)
    _, idx = nn.kneighbors(z)
    return idx[:, 1:].astype(np.int32)

def mknn_1d(ia, ib):
    return float((ia[:, :, None] == ib[:, None, :]).any(-1).mean())

def precompute_knn(emb, k):
    """Layerwise EEG: emb (n_layers, W, S, D) -> (n_layers, S, W, k)."""
    n_layers, W, Ns, _ = emb.shape
    idx = np.zeros((n_layers, Ns, W, k), dtype=np.int32)
    for l in range(n_layers):
        for s in range(Ns):
            idx[l, s] = knn_1d(l2(emb[l, :, s, :]), k)
    return idx

def precompute_knn_layers(emb, k):
    """Layerwise vision: emb (n_layers, W, D) -> (n_layers, W, k).
    A stale 2-D (W, D) array is promoted to a single layer for back-compat."""
    emb = np.asarray(emb)
    if emb.ndim == 2:
        emb = emb[None]
    n_layers, W, _ = emb.shape
    idx = np.zeros((n_layers, W, k), dtype=np.int32)
    for l in range(n_layers):
        idx[l] = knn_1d(l2(emb[l]), k)
    return idx

# ── Aristotelian MKNN ─────────────────────────────────────────────────────────
def _calibrate(T_obs, T_null, alpha):
    tau   = float(np.quantile(T_null, 1 - alpha))
    s_cal = max(T_obs - tau, 0.0) / (1 - tau) if tau < 1 else 0.0
    p_val = float((1 + (T_null >= T_obs).sum()) / (len(T_null) + 1))
    return tau, s_cal, p_val

def aristotelian_intra_subj(nn_a, nn_b, s, K=K_PERM, alpha=ALPHA, seed=42):
    """Intramodal, one subject. Searches all EEG-layer x EEG-layer pairs.
    nn_a, nn_b: (n_layers, S, W, k)."""
    LA, _, W, _ = nn_a.shape; LB = nn_b.shape[0]
    rng = np.random.default_rng(seed)
    a_s = nn_a[:, s]; b_s = nn_b[:, s]
    S_obs = np.array([[mknn_1d(a_s[la], b_s[lb]) for lb in range(LB)] for la in range(LA)])
    T_obs = float(S_obs.max())
    T_null = np.zeros(K)
    for ki in range(K):
        perm = rng.permutation(W); inv = np.argsort(perm)
        b_perm = inv[b_s[:, perm, :]]
        T_null[ki] = max(mknn_1d(a_s[la], b_perm[lb]) for la in range(LA) for lb in range(LB))
    tau, s_cal, p_val = _calibrate(T_obs, T_null, alpha)
    return T_obs, s_cal, tau, p_val

def aristotelian_cross(nn_eeg_layers, nn_vis_layers, K=K_PERM, alpha=ALPHA, seed=42):
    """Cross-modal, one subject. Searches all EEG-layer x vision-layer pairs.
    nn_eeg_layers: (LE, W, k)   nn_vis_layers: (LV, W, k)."""
    LE, W, _ = nn_eeg_layers.shape
    LV = nn_vis_layers.shape[0]
    rng = np.random.default_rng(seed)
    S_obs = np.array([[mknn_1d(nn_eeg_layers[le], nn_vis_layers[lv]) for lv in range(LV)]
                      for le in range(LE)])
    T_obs = float(S_obs.max())
    T_null = np.zeros(K)
    for ki in range(K):
        perm = rng.permutation(W); inv = np.argsort(perm)
        v_perm = inv[nn_vis_layers[:, perm, :]]            # (LV, W, k)
        T_null[ki] = max(mknn_1d(nn_eeg_layers[le], v_perm[lv])
                         for le in range(LE) for lv in range(LV))
    tau, s_cal, p_val = _calibrate(T_obs, T_null, alpha)
    return T_obs, s_cal, tau, p_val

# ── Generic cross-modal driver (used by all cross-modal compute cells) ────────
def run_cross_modal(eeg_model, vision_arch, vision_sizes, vfname, k_mknn):
    fam       = EEG[eeg_model]['family']
    eeg_sizes = EEG[eeg_model]['sizes']
    vis_nn = {}
    for vs in vision_sizes:
        vemb = load_npz(vfname(vs, fam))['embeddings'].astype(np.float32)
        print(f'  {vision_arch}-{vs} ({fam}): {vemb.shape}')
        vis_nn[vs] = precompute_knn_layers(vemb, k_mknn)   # (LV, W, k)
    results = {}
    for esz in eeg_sizes:
        emb = load_npz(EEG[eeg_model]['fname'](esz))['embeddings'].astype(np.float32)
        n_layers, W, S, _ = emb.shape
        print(f'  {eeg_model}-{esz}: {emb.shape}  precomputing EEG kNN...')
        nn_eeg = precompute_knn(emb, k_mknn)               # (n_layers, S, W, k)
        for vs in vision_sizes:
            LV = vis_nn[vs].shape[0]
            print(f'    cross  {eeg_model}-{esz} ({n_layers}L)  x  '
                  f'{vision_arch}-{vs} ({LV}L)  = {n_layers*LV} layer pairs')
            T_l, sc_l, tau_l, p_l = [], [], [], []
            for s in range(S):
                T, sc, tau, p = aristotelian_cross(
                    nn_eeg[:, s], vis_nn[vs], K=K_PERM, alpha=ALPHA, seed=42+s)
                T_l.append(T); sc_l.append(sc); tau_l.append(tau); p_l.append(p)
                print(f'      sub-{s+1:04d}: uncal={T:.4f}  tau={tau:.4f}  cal={sc:.4f}  p={p:.4f}')
            results[(esz, vs)] = dict(
                T_obs=np.array(T_l), s_cal=np.array(sc_l),
                tau=np.array(tau_l), p=np.array(p_l))
    return results

print('Setup complete. EEG models:', list(EEG.keys()))

## 1. Intramodal alignment per EEG model

For each EEG model (excluding reve): Aristotelian MKNN over **consecutive size pairs**,
per subject. Thin lines = subjects, bold line = mean.


In [ ]:
INTRA_MODELS = [m for m in EEG if m != 'reve']

intra_results = {}   # intra_results[model] = {(a,b): {'T_obs','s_cal','tau','p'}}

for model in INTRA_MODELS:
    sizes = EEG[model]['sizes']
    pairs = list(zip(sizes[:-1], sizes[1:]))
    print(f'\n=== {model}  sizes={sizes}  pairs={pairs} ===')
    nn_by_size = {}
    for sz in sizes:
        emb = load_npz(EEG[model]['fname'](sz))['embeddings'].astype(np.float32)
        print(f'  loading {model}-{sz}: {emb.shape}')
        nn_by_size[sz] = precompute_knn(emb, K_MKNN)
    S = next(iter(nn_by_size.values())).shape[1]
    intra_results[model] = {}
    for (a, b) in pairs:
        print(f'  pair {a}<->{b} ({nn_by_size[a].shape[0]}x{nn_by_size[b].shape[0]} layer pairs)')
        T_l, sc_l, tau_l, p_l = [], [], [], []
        for s in range(S):
            T, sc, tau, p = aristotelian_intra_subj(
                nn_by_size[a], nn_by_size[b], s, K=K_PERM, alpha=ALPHA, seed=42+s)
            T_l.append(T); sc_l.append(sc); tau_l.append(tau); p_l.append(p)
            print(f'    sub-{s+1:04d}: uncal={T:.4f}  tau={tau:.4f}  cal={sc:.4f}  p={p:.4f}')
        intra_results[model][(a, b)] = dict(
            T_obs=np.array(T_l), s_cal=np.array(sc_l),
            tau=np.array(tau_l), p=np.array(p_l))


In [ ]:
# ── Plot: one subplot per model ──────────────────────────────────────────────
ncols = 2
nrows = (len(INTRA_MODELS) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 6, nrows * 4.4))
axes_flat = axes.flatten()

for ax, model in zip(axes_flat, INTRA_MODELS):
    pairs = list(intra_results[model].keys())
    pair_labels = [f'{fmt_size(model, a)}\n<->\n{fmt_size(model, b)}' for (a, b) in pairs]
    x = np.arange(len(pairs))
    S = len(intra_results[model][pairs[0]]['s_cal'])
    cal_mat   = np.array([[intra_results[model][p]['s_cal'][s] for p in pairs] for s in range(S)])
    uncal_mat = np.array([[intra_results[model][p]['T_obs'][s] for p in pairs] for s in range(S)])
    for s in range(S):
        ax.plot(x, cal_mat[s],   color='steelblue',  alpha=0.25, lw=1, marker='o', markersize=3)
        ax.plot(x, uncal_mat[s], color='darkorange', alpha=0.25, lw=1, ls='--', marker='D', markersize=3)
    ax.plot(x, cal_mat.mean(0),   color='steelblue',  lw=2.5, marker='o', markersize=7,
            label='calibrated (s_cal)')
    ax.plot(x, uncal_mat.mean(0), color='darkorange', lw=2.2, ls='--', marker='D', markersize=7,
            label='uncalibrated (T_obs)')
    ax.set_xticks(x); ax.set_xticklabels(pair_labels, fontsize=9)
    ax.set_ylabel('mKNN score', fontsize=10)
    ax.set_title(DISPLAY[model], fontsize=13, fontweight='bold')
    ax.set_ylim(bottom=0)
    ax.grid(axis='y', alpha=0.3)
    ax.legend(fontsize=8)

for i in range(len(INTRA_MODELS), len(axes_flat)):
    axes_flat[i].set_visible(False)

fig.suptitle(f'Intramodal MKNN (k={K_MKNN}, K_perm={K_PERM}) — thin = subjects, thick = mean',
             fontsize=12)
plt.tight_layout()
plt.show()


## 2. Cross-modal alignment: EEG vs corresponding VideoMAE

One subplot per EEG model (all 5 incl. reve). Each EEG model is paired with the
VideoMAE embeddings of its stimulus family (`eeg_family`). For every (EEG layer,
VideoMAE layer) pair the per-subject mKNN is computed; the reported score is the
maximum over all layer pairs, calibrated against an aggregation-aware permutation
null. Within each subplot: x = EEG sizes, one colored bundle of lines per
VideoMAE size; thin lines = subjects, thick = mean; solid = calibrated,
dashed = uncalibrated.

In [ ]:
cross_vm = {}
for model in EEG:
    print(f'\n=== cross-modal {model} vs VideoMAE  (k={K_MKNN}) ===')
    cross_vm[model] = run_cross_modal(model, 'videomae', VIDEOMAE_SIZES, vmpath, K_MKNN)


In [ ]:
# ── Plot ─────────────────────────────────────────────────────────────────────
models = list(EEG.keys())
ncols = 3
nrows = (len(models) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 5.2, nrows * 4.4))
axes_flat = axes.flatten()

for ax, model in zip(axes_flat, models):
    eeg_sizes = EEG[model]['sizes']
    x = np.arange(len(eeg_sizes))
    for vs in VIDEOMAE_SIZES:
        color = vm_colors[vs]
        S = len(cross_vm[model][(eeg_sizes[0], vs)]['s_cal'])
        cal_mat   = np.array([[cross_vm[model][(esz, vs)]['s_cal'][s] for esz in eeg_sizes]
                              for s in range(S)])
        uncal_mat = np.array([[cross_vm[model][(esz, vs)]['T_obs'][s] for esz in eeg_sizes]
                              for s in range(S)])
        for s in range(S):
            ax.plot(x, cal_mat[s],   color=color, alpha=0.22, lw=1, marker='o', markersize=3)
            ax.plot(x, uncal_mat[s], color=color, alpha=0.18, lw=1, ls='--', marker='D', markersize=3)
        ax.plot(x, cal_mat.mean(0),   color=color, lw=2.5, marker='o', markersize=7,
                label=f'VideoMAE-{vs}  cal')
        ax.plot(x, uncal_mat.mean(0), color=color, lw=2.0, ls='--', marker='D', markersize=7,
                label=f'VideoMAE-{vs}  uncal')
    xtick_labels = [f'{sz} ({PARAMS[model][sz]}M)' for sz in eeg_sizes]
    ax.set_xticks(x); ax.set_xticklabels(xtick_labels, fontsize=9)
    ax.set_xlabel(f'{DISPLAY[model]} size')
    ax.set_ylabel('mKNN score', fontsize=10)
    ax.set_title(DISPLAY[model], fontsize=13, fontweight='bold')
    ax.set_ylim(bottom=0)
    ax.grid(axis='y', alpha=0.3)
    ax.legend(fontsize=7, ncol=2)

for i in range(len(models), len(axes_flat)):
    axes_flat[i].set_visible(False)

fig.suptitle(f'Cross-modal MKNN — EEG vs VideoMAE  (k={K_MKNN}, K_perm={K_PERM})  '
             '— solid = calibrated, dashed = uncalibrated',
             fontsize=12)
plt.tight_layout()
plt.show()


### Same, with k=10

Repeats the cross-modal MKNN for VideoMAE using k=10 nearest neighbours per point.


In [ ]:
cross_vm_k10 = {}
for model in EEG:
    print(f'\n=== cross-modal {model} vs VideoMAE  (k=10) ===')
    cross_vm_k10[model] = run_cross_modal(model, 'videomae', VIDEOMAE_SIZES, vmpath, 10)


In [ ]:
# ── Plot ─────────────────────────────────────────────────────────────────────
models = list(EEG.keys())
ncols = 3
nrows = (len(models) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 5.2, nrows * 4.4))
axes_flat = axes.flatten()

for ax, model in zip(axes_flat, models):
    eeg_sizes = EEG[model]['sizes']
    x = np.arange(len(eeg_sizes))
    for vs in VIDEOMAE_SIZES:
        color = vm_colors[vs]
        S = len(cross_vm_k10[model][(eeg_sizes[0], vs)]['s_cal'])
        cal_mat   = np.array([[cross_vm_k10[model][(esz, vs)]['s_cal'][s] for esz in eeg_sizes]
                              for s in range(S)])
        uncal_mat = np.array([[cross_vm_k10[model][(esz, vs)]['T_obs'][s] for esz in eeg_sizes]
                              for s in range(S)])
        for s in range(S):
            ax.plot(x, cal_mat[s],   color=color, alpha=0.22, lw=1, marker='o', markersize=3)
            ax.plot(x, uncal_mat[s], color=color, alpha=0.18, lw=1, ls='--', marker='D', markersize=3)
        ax.plot(x, cal_mat.mean(0),   color=color, lw=2.5, marker='o', markersize=7,
                label=f'VideoMAE-{vs}  cal')
        ax.plot(x, uncal_mat.mean(0), color=color, lw=2.0, ls='--', marker='D', markersize=7,
                label=f'VideoMAE-{vs}  uncal')
    xtick_labels = [f'{sz} ({PARAMS[model][sz]}M)' for sz in eeg_sizes]
    ax.set_xticks(x); ax.set_xticklabels(xtick_labels, fontsize=9)
    ax.set_xlabel(f'{DISPLAY[model]} size')
    ax.set_ylabel('mKNN score', fontsize=10)
    ax.set_title(DISPLAY[model], fontsize=13, fontweight='bold')
    ax.set_ylim(bottom=0)
    ax.grid(axis='y', alpha=0.3)
    ax.legend(fontsize=7, ncol=2)

for i in range(len(models), len(axes_flat)):
    axes_flat[i].set_visible(False)

fig.suptitle(f'Cross-modal MKNN — EEG vs VideoMAE  (k={10}, K_perm={K_PERM})  '
             '— solid = calibrated, dashed = uncalibrated',
             fontsize=12)
plt.tight_layout()
plt.show()


## 3. Cross-modal alignment: EEG vs corresponding DINOv2

Same structure as the VideoMAE section, with DINOv2 (4 sizes; CLS-token features
per layer) as the vision side. Score = max over all (EEG layer, DINOv2 layer)
pairs, aggregation-aware calibrated.

In [ ]:
cross_dn = {}
for model in EEG:
    print(f'\n=== cross-modal {model} vs DINOv2  (k={K_MKNN}) ===')
    cross_dn[model] = run_cross_modal(model, 'dinov2', DINOV2_SIZES, dnpath, K_MKNN)


In [ ]:
# ── Plot ─────────────────────────────────────────────────────────────────────
models = list(EEG.keys())
ncols = 3
nrows = (len(models) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 5.2, nrows * 4.4))
axes_flat = axes.flatten()

for ax, model in zip(axes_flat, models):
    eeg_sizes = EEG[model]['sizes']
    x = np.arange(len(eeg_sizes))
    for vs in DINOV2_SIZES:
        color = dn_colors[vs]
        S = len(cross_dn[model][(eeg_sizes[0], vs)]['s_cal'])
        cal_mat   = np.array([[cross_dn[model][(esz, vs)]['s_cal'][s] for esz in eeg_sizes]
                              for s in range(S)])
        uncal_mat = np.array([[cross_dn[model][(esz, vs)]['T_obs'][s] for esz in eeg_sizes]
                              for s in range(S)])
        for s in range(S):
            ax.plot(x, cal_mat[s],   color=color, alpha=0.22, lw=1, marker='o', markersize=3)
            ax.plot(x, uncal_mat[s], color=color, alpha=0.18, lw=1, ls='--', marker='D', markersize=3)
        ax.plot(x, cal_mat.mean(0),   color=color, lw=2.2, marker='o', markersize=6,
                label=f'DINOv2-{vs}  cal')
        ax.plot(x, uncal_mat.mean(0), color=color, lw=1.8, ls='--', marker='D', markersize=6,
                label=f'DINOv2-{vs}  uncal')
    xtick_labels = [f'{sz} ({PARAMS[model][sz]}M)' for sz in eeg_sizes]
    ax.set_xticks(x); ax.set_xticklabels(xtick_labels, fontsize=9)
    ax.set_xlabel(f'{DISPLAY[model]} size')
    ax.set_ylabel('mKNN score', fontsize=10)
    ax.set_title(DISPLAY[model], fontsize=13, fontweight='bold')
    ax.set_ylim(bottom=0)
    ax.grid(axis='y', alpha=0.3)
    ax.legend(fontsize=6, ncol=2)

for i in range(len(models), len(axes_flat)):
    axes_flat[i].set_visible(False)

fig.suptitle(f'Cross-modal MKNN — EEG vs DINOv2  (k={K_MKNN}, K_perm={K_PERM})  '
             '— solid = calibrated, dashed = uncalibrated',
             fontsize=12)
plt.tight_layout()
plt.show()


### Same, with k=10

Repeats the cross-modal MKNN for DINOv2 using k=10 nearest neighbours per point.


In [ ]:
cross_dn_k10 = {}
for model in EEG:
    print(f'\n=== cross-modal {model} vs DINOv2  (k=10) ===')
    cross_dn_k10[model] = run_cross_modal(model, 'dinov2', DINOV2_SIZES, dnpath, 10)


In [ ]:
# ── Plot ─────────────────────────────────────────────────────────────────────
models = list(EEG.keys())
ncols = 3
nrows = (len(models) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 5.2, nrows * 4.4))
axes_flat = axes.flatten()

for ax, model in zip(axes_flat, models):
    eeg_sizes = EEG[model]['sizes']
    x = np.arange(len(eeg_sizes))
    for vs in DINOV2_SIZES:
        color = dn_colors[vs]
        S = len(cross_dn_k10[model][(eeg_sizes[0], vs)]['s_cal'])
        cal_mat   = np.array([[cross_dn_k10[model][(esz, vs)]['s_cal'][s] for esz in eeg_sizes]
                              for s in range(S)])
        uncal_mat = np.array([[cross_dn_k10[model][(esz, vs)]['T_obs'][s] for esz in eeg_sizes]
                              for s in range(S)])
        for s in range(S):
            ax.plot(x, cal_mat[s],   color=color, alpha=0.22, lw=1, marker='o', markersize=3)
            ax.plot(x, uncal_mat[s], color=color, alpha=0.18, lw=1, ls='--', marker='D', markersize=3)
        ax.plot(x, cal_mat.mean(0),   color=color, lw=2.2, marker='o', markersize=6,
                label=f'DINOv2-{vs}  cal')
        ax.plot(x, uncal_mat.mean(0), color=color, lw=1.8, ls='--', marker='D', markersize=6,
                label=f'DINOv2-{vs}  uncal')
    xtick_labels = [f'{sz} ({PARAMS[model][sz]}M)' for sz in eeg_sizes]
    ax.set_xticks(x); ax.set_xticklabels(xtick_labels, fontsize=9)
    ax.set_xlabel(f'{DISPLAY[model]} size')
    ax.set_ylabel('mKNN score', fontsize=10)
    ax.set_title(DISPLAY[model], fontsize=13, fontweight='bold')
    ax.set_ylim(bottom=0)
    ax.grid(axis='y', alpha=0.3)
    ax.legend(fontsize=6, ncol=2)

for i in range(len(models), len(axes_flat)):
    axes_flat[i].set_visible(False)

fig.suptitle(f'Cross-modal MKNN — EEG vs DINOv2  (k={10}, K_perm={K_PERM})  '
             '— solid = calibrated, dashed = uncalibrated',
             fontsize=12)
plt.tight_layout()
plt.show()


## 4. Cross-modal alignment: EEG vs corresponding V-JEPA 2

Same structure as the VideoMAE and DINOv2 sections, with V-JEPA 2 (3 sizes: large,
huge, giant) as the video encoder. Score = max over all (EEG layer, V-JEPA 2 layer)
pairs, aggregation-aware calibrated against a permutation null.

In [ ]:
cross_vj2 = {}
for model in EEG:
    print(f'\n=== cross-modal {model} vs V-JEPA 2  (k={K_MKNN}) ===')
    cross_vj2[model] = run_cross_modal(model, 'vjepa2', VJEPA2_SIZES, vj2path, K_MKNN)

In [ ]:
# ── Plot ─────────────────────────────────────────────────────────────────────
models = list(EEG.keys())
ncols = 3
nrows = (len(models) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 5.2, nrows * 4.4))
axes_flat = axes.flatten()

for ax, model in zip(axes_flat, models):
    eeg_sizes = EEG[model]['sizes']
    x = np.arange(len(eeg_sizes))
    for vs in VJEPA2_SIZES:
        color = vj2_colors[vs]
        S = len(cross_vj2[model][(eeg_sizes[0], vs)]['s_cal'])
        cal_mat   = np.array([[cross_vj2[model][(esz, vs)]['s_cal'][s] for esz in eeg_sizes]
                              for s in range(S)])
        uncal_mat = np.array([[cross_vj2[model][(esz, vs)]['T_obs'][s] for esz in eeg_sizes]
                              for s in range(S)])
        for s in range(S):
            ax.plot(x, cal_mat[s],   color=color, alpha=0.22, lw=1, marker='o', markersize=3)
            ax.plot(x, uncal_mat[s], color=color, alpha=0.18, lw=1, ls='--', marker='D', markersize=3)
        ax.plot(x, cal_mat.mean(0),   color=color, lw=2.5, marker='o', markersize=7,
                label=f'V-JEPA2-{vs}  cal')
        ax.plot(x, uncal_mat.mean(0), color=color, lw=2.0, ls='--', marker='D', markersize=7,
                label=f'V-JEPA2-{vs}  uncal')
    xtick_labels = [f'{sz} ({PARAMS[model][sz]}M)' for sz in eeg_sizes]
    ax.set_xticks(x); ax.set_xticklabels(xtick_labels, fontsize=9)
    ax.set_xlabel(f'{DISPLAY[model]} size')
    ax.set_ylabel('mKNN score', fontsize=10)
    ax.set_title(DISPLAY[model], fontsize=13, fontweight='bold')
    ax.set_ylim(bottom=0)
    ax.grid(axis='y', alpha=0.3)
    ax.legend(fontsize=7, ncol=2)

for i in range(len(models), len(axes_flat)):
    axes_flat[i].set_visible(False)

fig.suptitle(f'Cross-modal MKNN — EEG vs V-JEPA 2  (k={K_MKNN}, K_perm={K_PERM})  '
             '— solid = calibrated, dashed = uncalibrated',
             fontsize=12)
plt.tight_layout()
plt.show()

### Same, with k=10

Repeats the cross-modal MKNN for V-JEPA 2 using k=10 nearest neighbours per point.

In [ ]:
cross_vj2_k10 = {}
for model in EEG:
    print(f'\n=== cross-modal {model} vs V-JEPA 2  (k=10) ===')
    cross_vj2_k10[model] = run_cross_modal(model, 'vjepa2', VJEPA2_SIZES, vj2path, 10)

In [ ]:
# ── Plot ─────────────────────────────────────────────────────────────────────
models = list(EEG.keys())
ncols = 3
nrows = (len(models) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 5.2, nrows * 4.4))
axes_flat = axes.flatten()

for ax, model in zip(axes_flat, models):
    eeg_sizes = EEG[model]['sizes']
    x = np.arange(len(eeg_sizes))
    for vs in VJEPA2_SIZES:
        color = vj2_colors[vs]
        S = len(cross_vj2_k10[model][(eeg_sizes[0], vs)]['s_cal'])
        cal_mat   = np.array([[cross_vj2_k10[model][(esz, vs)]['s_cal'][s] for esz in eeg_sizes]
                              for s in range(S)])
        uncal_mat = np.array([[cross_vj2_k10[model][(esz, vs)]['T_obs'][s] for esz in eeg_sizes]
                              for s in range(S)])
        for s in range(S):
            ax.plot(x, cal_mat[s],   color=color, alpha=0.22, lw=1, marker='o', markersize=3)
            ax.plot(x, uncal_mat[s], color=color, alpha=0.18, lw=1, ls='--', marker='D', markersize=3)
        ax.plot(x, cal_mat.mean(0),   color=color, lw=2.5, marker='o', markersize=7,
                label=f'V-JEPA2-{vs}  cal')
        ax.plot(x, uncal_mat.mean(0), color=color, lw=2.0, ls='--', marker='D', markersize=7,
                label=f'V-JEPA2-{vs}  uncal')
    xtick_labels = [f'{sz} ({PARAMS[model][sz]}M)' for sz in eeg_sizes]
    ax.set_xticks(x); ax.set_xticklabels(xtick_labels, fontsize=9)
    ax.set_xlabel(f'{DISPLAY[model]} size')
    ax.set_ylabel('mKNN score', fontsize=10)
    ax.set_title(DISPLAY[model], fontsize=13, fontweight='bold')
    ax.set_ylim(bottom=0)
    ax.grid(axis='y', alpha=0.3)
    ax.legend(fontsize=7, ncol=2)

for i in range(len(models), len(axes_flat)):
    axes_flat[i].set_visible(False)

fig.suptitle(f'Cross-modal MKNN — EEG vs V-JEPA 2  (k={10}, K_perm={K_PERM})  '
             '— solid = calibrated, dashed = uncalibrated',
             fontsize=12)
plt.tight_layout()
plt.show()